# 准备数据

In [1]:
# 过滤Alphalens的warning
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# 加载模块
import polars as pl

from vnpy.trader.constant import Interval

from vnpy.alpha import AlphaLab

In [3]:
# 创建数据中心
lab: AlphaLab = AlphaLab("./lab/csi300")

In [4]:
# 设置任务参数
name = "300_mlp"
index_symbol: str = "000300.SSE"
start: str = "2007-01-01"
end: str = "2024-10-31"
interval: Interval = Interval.DAILY
extended_days: int = 100

In [5]:
# 加载所有成分股代码
component_symbols: list[str] = lab.load_component_symbols(index_symbol, start, end)

In [6]:
print(component_symbols)

['300230.SZSE', '002284.SZSE', '601718.SSE', '300473.SZSE', '300590.SZSE', '600340.SSE', '002163.SZSE', '301057.SZSE', '603665.SSE', '688708.SSE', '000410.SZSE', '300290.SZSE', '600149.SSE', '603311.SSE', '300191.SZSE', '300509.SZSE', '601113.SSE', '300890.SZSE', '688411.SSE', '600157.SSE', '002572.SZSE', '300144.SZSE', '000630.SZSE', '300786.SZSE', '301135.SZSE', '301227.SZSE', '300757.SZSE', '300667.SZSE', '300195.SZSE', '603588.SSE', '000417.SZSE', '603707.SSE', '605300.SSE', '600571.SSE', '601168.SSE', '002809.SZSE', '600067.SSE', '002932.SZSE', '688226.SSE', '603170.SSE', '002036.SZSE', '001296.SZSE', '002777.SZSE', '000570.SZSE', '603810.SSE', '301303.SZSE', '300594.SZSE', '688456.SSE', '688530.SSE', '002806.SZSE', '688125.SSE', '301178.SZSE', '301297.SZSE', '300988.SZSE', '300088.SZSE', '688570.SSE', '600423.SSE', '688118.SSE', '300244.SZSE', '600084.SSE', '601808.SSE', '688435.SSE', '688277.SSE', '001229.SZSE', '600127.SSE', '000558.SZSE', '600420.SSE', '600009.SSE', '600791.SS

# 特征计算

In [7]:
# 加载模块
from datetime import datetime
from functools import partial

from vnpy.trader.constant import Interval

from vnpy.alpha.dataset import (
    AlphaDataset,
    process_drop_na,
    process_robust_zscore_norm,
    process_fill_na,
    process_cs_rank_norm,
    to_datetime
)
from vnpy.alpha.dataset.datasets.alpha_158 import Alpha158

In [8]:
# 加载成分股数据
component_symbols=component_symbols[:500]
df: pl.DataFrame = lab.load_bar_df(component_symbols, interval, start, end, extended_days)

2025-10-31 01:48:22 File lab/csi300/daily/688708.SSE.parquet does not exist
2025-10-31 01:48:22 File lab/csi300/daily/688411.SSE.parquet does not exist
2025-10-31 01:48:22 File lab/csi300/daily/603210.SSE.parquet does not exist
2025-10-31 01:48:23 File lab/csi300/daily/603072.SSE.parquet does not exist
2025-10-31 01:48:25 File lab/csi300/daily/301665.SZSE.parquet does not exist
2025-10-31 01:48:25 File lab/csi300/daily/301595.SZSE.parquet does not exist


In [9]:
# 设置数据时间段
train_period: tuple[str, str] = ("2007-01-01", "2020-12-31")
valid_period: tuple[str, str] = ("2021-01-01", "2021-12-31")
test_period: tuple[str, str] = ("2022-01-01", "2024-10-31")

In [10]:
# 创建数据集对象
dataset: AlphaDataset = Alpha158(
    df,
    train_period=train_period,
    valid_period=valid_period,
    test_period=test_period,
)

/home/lai/test_v1/vnpy/vnpy/alpha/dataset/utility.py:15: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  last_name: str = lf.columns[-1]


In [11]:
# 添加数据预处理器
fit_start_time: datetime = to_datetime(train_period[0])
fit_end_time: datetime = to_datetime(train_period[1])
print(f"fit_start_time: {fit_start_time}")
print(f"fit_end_time: {fit_end_time}")
dataset.add_processor("infer", partial(process_robust_zscore_norm, fit_start_time=fit_start_time, fit_end_time=fit_end_time))
dataset.add_processor("infer", partial(process_fill_na, fill_value=0, fill_label=False))

dataset.add_processor("learn", partial(process_drop_na, names=["label"]))
dataset.add_processor("learn", partial(process_cs_rank_norm, names=["label"]))

fit_start_time: 2007-01-01 00:00:00
fit_end_time: 2020-12-31 00:00:00


In [12]:
# 收集指数成分过滤器
#filters: dict[str, list[str]] = lab.load_component_filters(index_symbol, start, end)

In [13]:
#print(filters)

In [ ]:
# 准备特征和标签数据
dataset.prepare_data({}, max_workers=15)

2025-10-31 01:48:26 开始计算 DataProxy 因子特征


  1%|          | 1/159 [00:06<16:06,  6.11s/it]

Feature calculation kmid took: 6.114343643188477 seconds


  1%|▏         | 2/159 [00:14<19:39,  7.51s/it]

Feature calculation klen took: 8.486368179321289 seconds


...

Feature calculation rank_10 took: 30.190518856048584 seconds


 42%|████▏     | 66/159 [10:49<46:09, 29.78s/it]

Feature calculation rank_20 took: 30.055922746658325 seconds


 42%|████▏     | 67/159 [11:20<46:10, 30.11s/it]

Feature calculation rank_30 took: 30.87439203262329 seconds


In [ ]:
# 特征表现分析
#dataset.show_feature_performance("ma_60")

In [ ]:
dataset.df.shape

In [ ]:
# 保存到文件缓存
lab.save_dataset(name, dataset)

# 模型训练

In [ ]:
# 加载模块
import numpy as np

from vnpy.alpha import Segment, AlphaDataset, AlphaModel

from vnpy.alpha.model.models.mlp_model import MlpModel

In [ ]:
# 从文件缓存加载
dataset: AlphaDataset = lab.load_dataset(name)

In [ ]:
dataset.df.shape

In [ ]:
# 创建模型对象
kwargs = {
    "input_size": 158,
    "hidden_sizes": (256,),
    "lr": 0.002,
    "optimizer": "adam",
    "n_epochs": 8000,
    "batch_size": 8192,
    "weight_decay": 0.0002,
    "seed": 42
}

model: AlphaModel = MlpModel(**kwargs)

In [ ]:
from vnpy.alpha import Segment
df_train = dataset.fetch_learn(Segment.TRAIN)
print("TRAIN shape:", df_train.shape)
print(df_train.head())
print("TRAIN unique symbols:", df_train["vt_symbol"].unique()[:20])
print("TRAIN dates:", df_train["datetime"].min(), df_train["datetime"].max())

In [ ]:
# 使用数据集训练模型
model.fit(dataset)

In [ ]:
# 查看模型细节
model.detail()

In [ ]:
# 保存模型
lab.save_model(name, model)

# 预测信号

In [ ]:
model: AlphaModel = lab.load_model(name)

In [ ]:
# 用模型在测试集上预测
pre: np.ndarray = model.predict(dataset, Segment.TEST)

# 加载测试集数据
df_t: pl.DataFrame = dataset.fetch_infer(Segment.TEST)

# 合并预测信号列
df_t = df_t.with_columns(pl.Series(pre).alias("signal"))

# 提取信号数据
signal: pl.DataFrame = df_t["datetime", "vt_symbol", "signal"]

In [ ]:
# 检查信号绩效
dataset.show_signal_performance(signal)

In [ ]:
# 保存信号数据
lab.save_signal(name, signal)

# 策略回测

In [ ]:
# 加载模块
import importlib
from datetime import datetime

from vnpy.alpha.strategy import BacktestingEngine

import vnpy.alpha.strategy.strategies.equity_demo_strategy as equity_demo_strategy

In [ ]:
# 重载策略类
importlib.reload(equity_demo_strategy)
EquityDemoStrategy = equity_demo_strategy.EquityDemoStrategy

In [ ]:
# 从文件加载信号数据
signal = lab.load_signal(name)

In [ ]:
# 创建回测引擎对象
engine = BacktestingEngine(lab)

# 设置回测参数
engine.set_parameters(
    vt_symbols=component_symbols,
    interval=Interval.DAILY,
    start=datetime(2022, 1, 1),
    end=datetime(2024, 10, 31),
    capital=100000000
)

# 添加策略实例
setting = {"top_k": 30, "n_drop": 3, "hold_thresh": 3}
engine.add_strategy(EquityDemoStrategy, setting, signal)

In [ ]:
# 执行回测任务
engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

In [ ]:
# 显示超额收益分析结果
engine.show_performance(benchmark_symbol=index_symbol)